In [1]:
import glob
import os
import pandas as pd

# Проверяем наличие папки data или текущей директории
data_dir = "data" if os.path.exists("data") else "."
csv_files = sorted(glob.glob(os.path.join(data_dir, "orders_2024_*.csv")))

# Считываем и объединяем все файлы с заказами
df_orders = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"Всего загружено строк заказов: {len(df_orders)}")
print(df_orders.head(3).to_string())

Всего загружено строк заказов: 15000
   order_id  user_id       order_datetime  product_id     price  quantity    status
0         1     1044  2024-01-03 18:39:26          26  14038.15         1  refunded
1         2     1079  2024-01-03 17:45:43          16   9172.60         1      paid
2         3     1041  2024-01-06 22:46:38          25   9035.26         2      paid


In [2]:
# 1. Считаем общее число позиций/единиц товара в разрезе каждого заказа
order_sizes = (
    df_orders.groupby(["user_id", "order_id"])["quantity"].sum().reset_index()
)

# 2. Агрегируем показатели по каждому пользователю:
# - средняя цена позиции (price)
# - среднее количество позиций в заказе (quantity)
user_stats = (
    df_orders.groupby("user_id")
    .agg(avg_item_price=("price", "mean"))
    .reset_index()
)

user_stats["avg_order_items"] = (
    order_sizes.groupby("user_id")["quantity"].mean().values
)

print("Средние показатели пользователей (первые 5 строк):")
print(user_stats.head().to_string())

Средние показатели пользователей (первые 5 строк):
   user_id  avg_item_price  avg_order_items
0     1001     9645.921299         2.564972
1     1002    10042.690976         2.284553
2     1003    10323.160513         2.493590
3     1004     9977.171565         2.585034
4     1005     9544.486099         2.453901


In [3]:
# Z-нормализация признаков: (значение - среднее) / стандартное отклонение
for col in ["avg_item_price", "avg_order_items"]:
  user_stats[f"{col}_z"] = (
      user_stats[col] - user_stats[col].mean()
  ) / user_stats[col].std()

print("Данные после Z-нормализации:")
print(user_stats.head().to_string())

Данные после Z-нормализации:
   user_id  avg_item_price  avg_order_items  avg_item_price_z  avg_order_items_z
0     1001     9645.921299         2.564972         -1.152779           0.734768
1     1002    10042.690976         2.284553         -0.333812          -2.635084
2     1003    10323.160513         2.493590          0.245101          -0.123044
3     1004     9977.171565         2.585034         -0.469050           0.975860
4     1005     9544.486099         2.453901         -1.362150          -0.599996


In [4]:
# Находим пользователя с самым высоким Z-score по средней цене
highest_user = user_stats.loc[user_stats["avg_item_price_z"].idxmax()]

# Находим пользователя с самым низким Z-score по средней цене
lowest_user = user_stats.loc[user_stats["avg_item_price_z"].idxmin()]

print("⭐ Пользователь с самым ВЫСОКИМ Z-значением средней цены:")
print(f"  User ID: {int(highest_user['user_id'])}")
print(f"  Z-score: {highest_user['avg_item_price_z']:.4f}")
print(f"  Средняя цена позиции: {highest_user['avg_item_price']:.2f} руб.\n")

print("⭐ Пользователь с самым НИЗКИМ Z-значением средней цены:")
print(f"  User ID: {int(lowest_user['user_id'])}")
print(f"  Z-score: {lowest_user['avg_item_price_z']:.4f}")
print(f"  Средняя цена позиции: {lowest_user['avg_item_price']:.2f} руб.")

⭐ Пользователь с самым ВЫСОКИМ Z-значением средней цены:
  User ID: 1040
  Z-score: 2.4996
  Средняя цена позиции: 11415.41 руб.

⭐ Пользователь с самым НИЗКИМ Z-значением средней цены:
  User ID: 1063
  Z-score: -2.0213
  Средняя цена позиции: 9225.16 руб.
